# Lab 1 — DVM Color Classification

Классификация цвета автомобиля по фронтальному фото (датасет DVM, top-6 цветов: `black, grey, white, blue, silver, red`).

**Цель:** обучить и сравнить 3 модели (1 from-scratch + 2 pretrained), получить F1_macro > 0.8.

**Запуск:** `Runtime → Change runtime type → GPU`, затем выполни ячейки по порядку. Все ячейки идемпотентны — переисполнение не ломает состояние.

## 1. Clone / Pull + path setup

Подтягиваем свежие `.py` с GitHub (`GITHUB_TOKEN` в Colab Secrets), переходим в папку лабы и добавляем её в `sys.path`.


In [ ]:
import os, sys
from google.colab import userdata

TOKEN = userdata.get("GITHUB_TOKEN")
REPO_DIR = "/content/ITMO-CV"
LAB_DIR = f"{REPO_DIR}/lab1-CLAS"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://{TOKEN}@github.com/Ma-XD/ITMO-CV.git {REPO_DIR}

os.chdir(LAB_DIR)
sys.path.insert(0, LAB_DIR)
print(f"📂 {os.getcwd()}")


## 2. Drive mount

Нужен только если хочешь сохранять `outputs/` (чекпоинты, графики) на Drive — иначе они остаются на локальном SSD VM и пропадут с сессией. Можно пропустить, если запускаешь без Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Зависимости

Ставим всё, что нужно лабе. `requirements.txt` лежит рядом с ноутбуком.


In [ ]:
!pip install -q -r requirements.txt


## 4. Verify окружения

In [ ]:
from env_config import print_env
print_env()

## 5. Загрузка датасета

Драйв медленный для случайного чтения мелких файлов (61k jpg). Поэтому архив **копируем** один раз с Drive на локальный SSD VM (`/content/`) и распаковываем туда же — обучение читает из `/content/data/dvm/...`, а на Drive ничего не пишем.

Источник: `MyDrive/ITMO-CV/lab1-CLAS/data/dvm_confirmed_fronts.zip` (765 МБ, без сжатия).

In [ ]:
%%bash
set -e
SRC=/content/drive/MyDrive/ITMO-CV/lab1-CLAS/data/dvm_confirmed_fronts.zip
DATA_DIR=/content/data

[ ! -f "$SRC" ] && echo "❌ Архив не найден: $SRC" && exit 1

if [ -d "$DATA_DIR/dvm/confirmed_fronts" ]; then
  echo "✅ Уже распаковано"
else
  echo "📥 Копируем архив..."
  cp "$SRC" /tmp/dvm.zip
  echo "📦 Распаковываем..."
  unzip -q /tmp/dvm.zip -d "$DATA_DIR"
  rm /tmp/dvm.zip
fi

echo "Изображений: $(find "$DATA_DIR/dvm/confirmed_fronts" -name '*.jpg' | wc -l)"


## 6. Сборка `index.csv`

Парсит filename'ы по `$$`, фильтрует `unlisted`/`multicolour` и off-target цвета, оставляет только top-6 → `/content/data/dvm/index.csv`. Идемпотентно: если CSV уже есть, скрипт его не перезаписывает (для пересборки — `--force`).

In [ ]:
!python build_index.py


## 7. Работа с датасетом

Что мы делаем с исходным `dvm_confirmed_fronts.zip` (61k jpg) перед обучением:

- **Фильтрация цветов** (`build_index.py`):
  - убираем `unlisted` — нет ground truth, шум для supervised-обучения;
  - убираем `multicolour` — мульти-метка, single-label classification неприменим;
  - оставляем top-6 (`black, grey, white, blue, silver, red`) — у остальных long tail (~сотни примеров на класс), мало для обучения и оценки.
- **Стратифицированный split** 70/15/15 (`data.py:make_splits`) — доли классов сохраняются во всех трёх сплитах.
- **Дисбаланс**: `black/grey/white` доминируют. В train используем `WeightedRandomSampler` (`data.py`), чтобы каждая эпоха видела классы примерно поровну.
- **Augmentations** (`data.py:build_transforms`): **НЕ** трогаем `hue`/`saturation` — таргет = цвет, перекрашивание сломает label. Только geometry (crop, flip) + brightness/contrast.


**Распределение классов после фильтра + годы/марки/модели.**

In [ ]:
from build_index import print_summary
from data import load_index

df = load_index()
print_summary(df)


**Стратифицированный split train/val/test и доли классов в каждом.**

In [ ]:
import pandas as pd

from config import TARGET_COLORS
from data import load_index, make_splits

splits = make_splits(load_index())
print(f"train: {len(splits.train)}")
print(f"val:   {len(splits.val)}")
print(f"test:  {len(splits.test)}")
print()

# Доли классов в каждом сплите — должны совпадать (стратифицированный split).
shares = pd.DataFrame({
    name: getattr(splits, name)["color"].value_counts(normalize=True).reindex(TARGET_COLORS) * 100
    for name in ["train", "val", "test"]
}).round(1)
print("Доля классов в каждом сплите (%):")
print(shares)


**Augmentations глазами: оригинал + 4 прохода `train_tf`.**

In [ ]:
import matplotlib.pyplot as plt
import torch
from PIL import Image

from config import IMAGENET_MEAN, IMAGENET_STD
from data import build_transforms

# Augmentations (train_tf): RandomResizedCrop + HFlip + brightness/contrast.
# hue/saturation НЕ трогаем — таргет = цвет, перекрашивание сломает label.
train_tf, _ = build_transforms()
img = Image.open(splits.train.iloc[0]["path"]).convert("RGB")

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(img); axes[0].set_title("оригинал"); axes[0].axis("off")
for i in range(1, 5):
    t = train_tf(img)
    show = (t * std + mean).clip(0, 1).permute(1, 2, 0).numpy()
    axes[i].imshow(show); axes[i].set_title(f"train_tf #{i}"); axes[i].axis("off")
plt.tight_layout()
plt.show()


## 8. Train: custom_resnet18 (from-scratch)

ResNet-18, написанный руками. Обучается с нуля и одной param-group (head + backbone вместе, `LR=1e-3`). Scheduler — `CosineAnnealingLR`. 12 эпох — столько же, сколько fine-tune, для честного сравнения по budget'у.

Веса сохранятся в `CHECKPOINT_DIR/custom_resnet18_last.pth` и подхватятся в разделе 12.


In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import MODEL_CUSTOM
from data import get_dataloaders, load_index, make_splits
from engine import fit
from env_config import get_device
from models import build_model, get_param_groups

EPOCHS = 12
LR = 1e-3
WEIGHT_DECAY = 1e-4
print(f"[{MODEL_CUSTOM}] epochs={EPOCHS}  lr={LR}  weight_decay={WEIGHT_DECAY}")

device = get_device()
splits = make_splits(load_index())
loaders = get_dataloaders(splits)

model = build_model(MODEL_CUSTOM).to(device)
optimizer = Adam(get_param_groups(model, MODEL_CUSTOM, lr_head=LR, weight_decay=WEIGHT_DECAY))
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history_custom = fit(
    model, loaders,
    model_name=MODEL_CUSTOM,
    epochs=EPOCHS,
    optimizer=optimizer,
    scheduler=scheduler,
)


## 9. Train: resnet18 (fine-tune ImageNet)

`torchvision.models.resnet18` с весами ImageNet. Заменён только последний `Linear` на 6 классов. Param groups: head — `LR=1e-3`, backbone — `LR=1e-4` (чтобы не сломать ImageNet-фичи). 12 эпох, CosineAnnealingLR.


In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import MODEL_RESNET18
from data import get_dataloaders, load_index, make_splits
from engine import fit
from env_config import get_device
from models import build_model, get_param_groups

EPOCHS = 12
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4   # backbone предобучен на ImageNet — низкий LR, чтобы не ломать фичи
WEIGHT_DECAY = 1e-4
print(f"[{MODEL_RESNET18}] epochs={EPOCHS}  lr_head={LR_HEAD}  lr_backbone={LR_BACKBONE}  weight_decay={WEIGHT_DECAY}")

device = get_device()
splits = make_splits(load_index())
loaders = get_dataloaders(splits)

model = build_model(MODEL_RESNET18).to(device)
optimizer = Adam(get_param_groups(
    model, MODEL_RESNET18,
    lr_head=LR_HEAD, lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
))
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history_resnet18 = fit(
    model, loaders,
    model_name=MODEL_RESNET18,
    epochs=EPOCHS,
    optimizer=optimizer,
    scheduler=scheduler,
)


## 10. Train: mobilenet_v3_small (fine-tune ImageNet)

Лёгкая модель из другого семейства (depthwise-separable conv, h-swish, SE) — хорошо для сравнения «разные архитектуры». Заменён только последний `Linear` в `classifier`. Та же стратегия param groups, 12 эпох.


In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import MODEL_MOBILENETV3
from data import get_dataloaders, load_index, make_splits
from engine import fit
from env_config import get_device
from models import build_model, get_param_groups

EPOCHS = 12
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4
WEIGHT_DECAY = 1e-4
print(f"[{MODEL_MOBILENETV3}] epochs={EPOCHS}  lr_head={LR_HEAD}  lr_backbone={LR_BACKBONE}  weight_decay={WEIGHT_DECAY}")

device = get_device()
splits = make_splits(load_index())
loaders = get_dataloaders(splits)

model = build_model(MODEL_MOBILENETV3).to(device)
optimizer = Adam(get_param_groups(
    model, MODEL_MOBILENETV3,
    lr_head=LR_HEAD, lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
))
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history_mobilenet = fit(
    model, loaders,
    model_name=MODEL_MOBILENETV3,
    epochs=EPOCHS,
    optimizer=optimizer,
    scheduler=scheduler,
)


## 11. Запуск инференса (ручной)

Проводим **одну** картинку через модель end-to-end, без `DataLoader`, чтобы видеть каждый шаг:

`PIL.Image` → `eval_tf` (resize + center-crop + normalize) → `unsqueeze(0)` (добавляем batch-dim) → `model.forward` → `softmax` → barplot вероятностей по 6 классам.

Используем pretrained `resnet18` — обычно у неё самые уверенные предсказания.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from config import IMAGENET_MEAN, IMAGENET_STD, MODEL_RESNET18, TARGET_COLORS
from data import build_transforms, load_index, make_splits
from engine import load_checkpoint
from env_config import get_device
from models import build_model

device = get_device()
splits = make_splits(load_index())
_, eval_tf = build_transforms()

# Берём pretrained resnet18 — обычно даёт самые уверенные предсказания.
model = build_model(MODEL_RESNET18).to(device)
load_checkpoint(model, MODEL_RESNET18, device=device)
model.eval()

# Шаг 1: одна картинка из test (можно подменить на свой файл).
row = splits.test.sample(n=1, random_state=0).iloc[0]
img = Image.open(row["path"]).convert("RGB")
true_color = TARGET_COLORS[int(row["label"])]
print(f"PIL image: {img.size}  (W x H)")

# Шаг 2: transform → тензор (3, 224, 224), нормализация ImageNet.
x = eval_tf(img)
print(f"after eval_tf: shape={tuple(x.shape)}  min={x.min():.2f}  max={x.max():.2f}")

# Шаг 3: добавляем batch-dim → (1, 3, 224, 224), forward.
batch = x.unsqueeze(0).to(device)
with torch.no_grad():
    logits = model(batch)
print(f"logits: shape={tuple(logits.shape)}  values={logits[0].cpu().numpy().round(2)}")

# Шаг 4: softmax → вероятности (sum=1), argmax → имя класса.
probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
pred_idx = int(probs.argmax())
print(f"\nGT:   {true_color}")
print(f"Pred: {TARGET_COLORS[pred_idx]}  (p={probs[pred_idx]:.3f})")

# Визуализация: картинка + barplot вероятностей по 6 классам.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.imshow(img); ax1.axis("off")
ax1.set_title(f"GT: {true_color}", fontsize=12)

colors = ["green" if i == pred_idx else "lightgray" for i in range(len(TARGET_COLORS))]
ax2.bar(TARGET_COLORS, probs, color=colors)
ax2.set_ylim(0, 1); ax2.set_ylabel("probability")
ax2.set_title(f"pred: {TARGET_COLORS[pred_idx]} ({probs[pred_idx]:.2f})")
for i, p in enumerate(probs):
    ax2.text(i, p + 0.02, f"{p:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


## 12. Инференс и оценка на test

Грузим обученные чекпоинты с диска, считаем метрики на test, рисуем confusion matrices и показываем предсказания на случайных тестовых картинках.

**Метрики на test для всех моделей** (accuracy, F1_macro). Чекпоинты грузим с диска.

In [ ]:
from config import ALL_MODELS, TARGET_COLORS
from data import get_dataloaders, load_index, make_splits
from engine import evaluate, load_checkpoint
from env_config import get_device
from models import build_model

device = get_device()
splits = make_splits(load_index())
loaders = get_dataloaders(splits)

results: dict[str, dict] = {}
for name in ALL_MODELS:
    model = build_model(name).to(device)
    meta = load_checkpoint(model, name, device=device)
    res = evaluate(model, loaders["test"], device=device, class_names=TARGET_COLORS)
    results[name] = res
    print(f"[{name}] acc={res['accuracy']:.4f}  f1_macro={res['f1_macro']:.4f}  (epochs trained: {meta['epoch']})")
    del model


**Confusion matrices** (нормированные по строкам — доля от истинных). По диагонали — корректные предсказания, вне — типичные путаницы.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from config import FIGURE_DIR, TARGET_COLORS

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4.5))
if len(results) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm = res["confusion_matrix"].astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{name}\nF1_macro = {res['f1_macro']:.3f}")
    ax.set_xticks(range(len(TARGET_COLORS)))
    ax.set_yticks(range(len(TARGET_COLORS)))
    ax.set_xticklabels(TARGET_COLORS, rotation=45, ha="right")
    ax.set_yticklabels(TARGET_COLORS)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm_norm[i, j] > 0.5 else "black"
            ax.text(j, i, f"{int(cm[i, j])}", ha="center", va="center", color=color, fontsize=8)

plt.tight_layout()
out = FIGURE_DIR / "confusion_matrices.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.show()
print(f"💾 {out}")


**Classification report**: precision / recall / F1 / support по каждому классу для каждой модели.

In [ ]:
from sklearn.metrics import classification_report

from config import TARGET_COLORS

for name, res in results.items():
    print(f"=== {name} ===")
    print(classification_report(
        res["y_true"], res["y_pred"],
        labels=list(range(len(TARGET_COLORS))),
        target_names=TARGET_COLORS,
        digits=3,
        zero_division=0,
    ))


**Где модель ошибается**: топ-12 самых уверенных ошибок `resnet18` на test (модель промахнулась с высокой `p_pred`). Самые наглядные кейсы — типичные путаницы `silver ↔ grey ↔ white` хорошо видны.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from config import FIGURE_DIR, IMAGENET_MEAN, IMAGENET_STD, MODEL_RESNET18, TARGET_COLORS
from data import build_transforms, load_index, make_splits
from engine import load_checkpoint
from env_config import get_device
from models import build_model

device = get_device()
splits = make_splits(load_index())
_, eval_tf = build_transforms()

# Берём уже посчитанные предсказания resnet18 из раздела 12 (evaluate).
res = results[MODEL_RESNET18]
wrong_idx = np.where(res["y_true"] != res["y_pred"])[0]
print(f"misclassified: {len(wrong_idx)} / {len(res['y_true'])} "
      f"({len(wrong_idx) / len(res['y_true']) * 100:.1f}%)")

# Прогоняем misclassified ещё раз, чтобы получить уверенность модели в (ошибочном) предсказании.
model = build_model(MODEL_RESNET18).to(device)
load_checkpoint(model, MODEL_RESNET18, device=device)
model.eval()

records: list[tuple[int, float, int]] = []
with torch.no_grad():
    for idx in wrong_idx:
        row = splits.test.iloc[int(idx)]
        x = eval_tf(Image.open(row["path"]).convert("RGB")).unsqueeze(0).to(device)
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        records.append((int(idx), float(probs.max()), int(probs.argmax())))

# Топ-12 самых уверенных ошибок (модель «уверенно» промахнулась — самые наглядные кейсы).
records.sort(key=lambda t: -t[1])
top = records[:12]

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, (idx, conf, pred_idx) in zip(axes.flat, top):
    row = splits.test.iloc[idx]
    x = eval_tf(Image.open(row["path"]).convert("RGB"))
    show = (x * std + mean).clip(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(show); ax.axis("off")
    true_color = TARGET_COLORS[int(row["label"])]
    pred_color = TARGET_COLORS[pred_idx]
    ax.set_title(f"GT: {true_color}\npred: {pred_color}  ({conf:.2f})",
                 color="red", fontsize=10)

fig.suptitle(f"Top-12 уверенных ошибок: {MODEL_RESNET18}", fontsize=14)
plt.tight_layout()
out = FIGURE_DIR / "misclassified_examples.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.show()
print(f"💾 {out}")


## 13. Сравнение моделей

Сводная таблица + кривые `val_f1_macro` по эпохам + per-class F1.


**Сводная таблица**: число параметров (M), test accuracy, test F1_macro и F1 по каждому классу. Сортировка по `F1_macro`.

In [ ]:
import pandas as pd

from models import build_model

rows = []
for name, res in results.items():
    n_params = sum(p.numel() for p in build_model(name).parameters())
    rows.append({
        "model": name,
        "params (M)": round(n_params / 1e6, 2),
        "test accuracy": round(res["accuracy"], 4),
        "test F1_macro": round(res["f1_macro"], 4),
        **{f"F1[{c}]": round(res["f1_per_class"][c], 3) for c in TARGET_COLORS},
    })

summary = pd.DataFrame(rows).sort_values("test F1_macro", ascending=False).reset_index(drop=True)
summary


**Кривые обучения**: train/val loss и val F1_macro по эпохам. Красная пунктирная линия — целевой `F1_macro = 0.8`.

In [ ]:
import json
import matplotlib.pyplot as plt

from config import LOG_DIR, ALL_MODELS, FIGURE_DIR

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

for name in ALL_MODELS:
    log_path = LOG_DIR / f"{name}_history.json"
    if not log_path.exists():
        print(f"⚠️  нет {log_path}")
        continue
    with log_path.open() as f:
        hist = json.load(f)["history"]
    epochs = range(1, len(hist["val_f1_macro"]) + 1)
    ax1.plot(epochs, hist["val_loss"], label=f"{name} val")
    ax1.plot(epochs, hist["train_loss"], label=f"{name} train", linestyle="--", alpha=0.6)
    ax2.plot(epochs, hist["val_f1_macro"], label=name, linewidth=2)

ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title("train/val loss"); ax1.legend(fontsize=8)
ax2.set_xlabel("epoch"); ax2.set_ylabel("val F1_macro"); ax2.set_title("validation F1_macro")
ax2.axhline(0.8, color="red", linestyle=":", label="target = 0.8")
ax2.legend()

plt.tight_layout()
out = FIGURE_DIR / "training_curves.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.show()
print(f"💾 {out}")
